# router_voto — Laya fina para el router del chat de deepseek-flow

**Qué hace:** fine-tuning completo (encoder incluido) del checkpoint *multilingual* de Laya con
`layaft` (RLCD + calibración de temperatura), sobre casos generados con DeepSeek y pasados por
juez ciego para la pregunta del router de deepseek-flow (¿esta pregunta necesita herramientas o
se responde directa?). Al final mide el test set de 24 casos escritos a mano, jamás entrenados
(la task replica byte a byte la pregunta `PREGUNTA_ROUTER` de producción; el estado es
`{"pregunta": ...}` como en producción).

**Entrada:** el dataset `roquedb/flow-router-voto` (task YAML + contexts + test + datos verificados).
**Acelerador:** GPU T4 (layaft usa una). **Internet:** sí, para pip y para el dataset del base.

**Cómo reproducirlo local:** `train/generate_data.py generate n=70 context=all task=train/tasks/router_voto.yaml`
y `train/generate_data.py verify task=train/tasks/router_voto.yaml` en el repo de bmo.

In [1]:
!nvidia-smi
%pip install -q laya-finetune==0.1.2 pyyaml
import layaft, laya, torch
print("layaft listo | laya", laya.__version__, "| torch", torch.__version__, "| GPUs:", torch.cuda.device_count())

Tue Oct  6 00:29:49 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import glob, os, shutil, pathlib
W = pathlib.Path("/kaggle/working")
for name in ("tasks/router_voto.yaml", "tasks/contexts_router.txt", "tasks/router_voto_test.jsonl", "data/router_voto_verified.jsonl"):
    hits = glob.glob(f"/kaggle/input/**/{os.path.basename(name)}", recursive=True)
    assert hits, f"no encontré {name}: fijate que el dataset flow-router-voto esté agregado como Input"
    dest = W / name
    dest.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy(hits[0], dest)
print("listo:", sorted(str(p.relative_to(W)) for p in W.rglob("*") if p.is_file()))

listo: ['__notebook__.ipynb', 'data/router_voto_verified.jsonl', 'tasks/contexts_router.txt', 'tasks/router_voto.yaml', 'tasks/router_voto_test.jsonl']


In [3]:
# Dónde está el base: el directorio de input que contiene model.safetensors (no confiar en el slug de mount)
import pathlib
candidatos = [p for p in pathlib.Path("/kaggle/input").rglob("model.safetensors")]
print("inputs:", [str(p) for p in pathlib.Path("/kaggle/input").glob("*")])
assert candidatos, "no encontré el checkpoint base: revisá que bmo-laya-base esté como Input"
BASE = str(candidatos[0].parent)
print("base:", BASE)
# Entrenamiento desde el base local; ctx por defecto 1k -> runs/router_voto-1k
!layaft train task=tasks/router_voto.yaml data=data/router_voto_verified.jsonl model={BASE} profile=full

inputs: ['/kaggle/input/datasets']
base: /kaggle/input/datasets/roquedb/bmo-laya-base
558 cases from data/router_voto_verified.jsonl
Teacher: no TYPESAFE_API_KEY or OPENROUTER_API_KEY; 558 uncached cases use the smoothed label
/kaggle/input/datasets/roquedb/bmo-laya-base · ctx 1024 · attention sdpa
446 sequences · tokens: median 114, max 153
322 M trainable parameters · 8 updates · torch.float16
Validation CE before training: 1.1480
/usr/local/lib/python3.13/dist-packages/layaft/train/rlcd.py:136: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scheduler.step()
epoch 1/4: train CE 1.1176 · validation CE 1.1480 · 6 s · max VRAM 3.26 GB
epoch 2/4: tr

In [4]:
# Validación en el test set escrito a mano (el número que cuenta)
import pathlib
runs = sorted(p for p in pathlib.Path("runs").glob("*"))
print("runs:", [str(r) for r in runs])
RUN = str(runs[0])
!layaft val task=tasks/router_voto.yaml model={RUN}

runs: ['runs/router_voto-1k']
| | runs/router_voto-1k |
|---|---|
| confirma_herramientas | 27/30 |
| confirma_herramientas ECE | 0.119 |
| All answers right | 27/30 |
| Latency p50 | 32 ms |
Saved in /kaggle/working/runs/val/router_voto-router_voto-1k


In [5]:
import glob, pathlib
for f in sorted(pathlib.Path("runs/val").rglob("*")):
    if f.is_file():
        print("=" * 20, f, "=" * 20)
        print(f.read_text(encoding="utf-8")[:2000])

==================== runs/val/router_voto-router_voto-1k/chart.svg ====================
<svg xmlns="http://www.w3.org/2000/svg" width="880" height="440" viewBox="0 0 880 440" role="img" aria-label="Accuracy per question">
<rect x="4" y="4" width="875" height="435" rx="12" fill="#0d0d0d"/>
<rect x="1" y="1" width="875" height="435" rx="12" fill="white" stroke="#0d0d0d" stroke-width="2"/>
<text x="24" y="36" text-anchor="start" font-family="Nunito,system-ui,sans-serif" font-size="20" font-weight="700" fill="#0d0d0d">Accuracy per question</text>
<text x="24" y="57" text-anchor="start" font-family="Nunito,system-ui,sans-serif" font-size="12" font-weight="400" fill="#696969">30 hand-written test cases, never trained on</text>
<rect x="72" y="74" width="15" height="15" rx="3" fill="#4fa8f0" stroke="#0d0d0d" stroke-width="2"/>
<text x="95" y="87" text-anchor="start" font-family="Nunito,system-ui,sans-serif" font-size="12" font-weight="700" fill="#0d0d0d">runs/router_voto-1k</text>
<line x1="7

## Para traerlo a casa

El checkpoint queda en `runs/router_voto-1k/` (pesos + tokenizer + config con temperatura
calibrada). En la máquina local:

```
kaggle kernels output roquedb/router-voto -p train/runs-kaggle/
```

y en el `.env` de deepseek-flow: `LAYA_MODEL=<ruta>/train/runs-kaggle/runs/router_voto-1k`.